### Import packages etc.


In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_links, fetch_website_contents
from IPython.display import Markdown, display, update_display


In [2]:
import gradio as gr


In [3]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set
Google API Key exists and begins AIzaSyDE


In [4]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

# anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)


### Writing the necessary prompts for the AI model later


In [5]:
link_system_prompt = """
You are given a list of links from a cooking website. Pick the 3 recipes easiest for a COMPLETE BEGINNER, ranked by (in order):
1. Short total time (ideally <30 min)
2. Only basic equipment (pan, pot, oven, knife, stovetop — no stand mixer, food processor, thermometer, pressure cooker, sous vide, etc.)
3. Few, common ingredients
4. Simple techniques (no deboning, folding, tempering, proofing, laminating, multi-stage cooking)

Ignore non-recipe links (category, blog, about, shop, login, legal, email).

Respond in JSON:
{"recipes":[
  {"rank":1,"title":"One-Pan Garlic Butter Pasta","url":"https://...","reason":"15 min, one pan, 5 ingredients"},
  {"rank":2,"title":"Scrambled Eggs on Toast","url":"https://...","reason":"10 min, basic pan, no knife skills"},
  {"rank":3,"title":"Microwave Mug Oatmeal","url":"https://...","reason":"3 min, microwave, 3 ingredients"}
]}
"""


In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the cooking website {url} -
Please decide the 3 EASIEST recipes for a complete beginner to make, prioritising fast cook time and minimal kitchen equipment. Respond with the full https URL in JSON format.
Do not include category pages, blog posts that aren't recipes, login/register pages, Terms of Service, Privacy, or email links.
Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [7]:
def select_relevant_links(url, model):
    print(f"Selecting relevant links for {url} by calling {model}")
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['recipes'])} relevant links")
    return links


### Format the user prompt to pass into the AI


In [11]:
# format the user prompt first.
def fetch_page_and_all_relevant_links(url, model):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url, model)
    result = f"## Landing Page:\n\n{contents}\n## Top 3 Recipes:\n"
    for link in relevant_links["recipes"]:
        result += f"\n\n### Recipe name: {link['title']}\n"
        result += f"\n\n### Reason for selection: {link['reason']}\n"
        result += fetch_website_contents(link["url"])
    return result


In [10]:
cookbook_system_prompt = """
You produce a short, friendly "Beginner's Cookbook" in markdown from 3 pre-selected easy recipes (raw scraped content supplied).

For EACH recipe, output a section with:
- A one-line confidence-boosting intro
- **Total time** and **servings**
- **Equipment** (basics only)
- **Ingredients** as bullets, with common substitutions in brackets
- **Steps**: numbered, plain encouraging language; define any cooking term on first use (e.g. "sauté = cook in a little oil over medium heat until soft")
- **Beginner tip**: the single most common mistake to avoid

End with "What to cook first" — pick ONE recipe as the best starting point, one-sentence reason.

Be warm, concise, never condescending. Do NOT invent ingredients or steps beyond the scraped content — only rephrase and clarify.
"""


In [12]:
def get_cookbook_user_prompt(url, model):
    user_prompt = f"""
Here are 3 beginner-friendly recipes scraped from {url}. Please turn them into a short Beginner's Cookbook in markdown, following the format in your system prompt.
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url, model)
    user_prompt = user_prompt[
        :5_000
    ]  # Truncate if more than 5,000 characters (can use 5000, 5_000 is for readability)
    return user_prompt


In [14]:
get_cookbook_user_prompt("https://www.budgetbytes.com/", "gpt-4.1-mini")


Selecting relevant links for https://www.budgetbytes.com/ by calling gpt-4.1-mini
Found 3 relevant links


"\nHere are 3 beginner-friendly recipes scraped from https://www.budgetbytes.com/. Please turn them into a short Beginner's Cookbook in markdown, following the format in your system prompt.\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome - Budget Bytes\n\nSkip to content\nSign up\nfor access to our weekly meal plans\nSign Up\nLogin\nRecipes\nRecipe Search\nCategories\nIngredient Index\nSurprise Me\nRecipe Roundups\nPopular\nMeal Prep\nDinner Ideas\nOne Pot Meals\nChicken Recipes\nPasta Recipes\nVegetarian Recipes\nSalad Recipes\nBreakfast Recipes\nRecipes under $10\nSandwich Recipes\nSoup Recipes\nAir Fryer Recipes\nMeal Plans\nAbout\nAbout Budget Bytes\nNew? Start Here\nFAQ\nContact\nSearch for\nLatest Recipes\nChicken Kabobs\n$12.57 recipe / $2.51 serving\nGnocchi with Spring Vegetables\n$8.65 recipe / $2.16 serving\nQuick And Easy Weeknight Dinner Ideas\nMore Recent Recipes\n1700+\naffordable & delicious r

In [18]:
def stream_cookbook(url, model):
    stream = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": cookbook_system_prompt},
            {"role": "user", "content": get_cookbook_user_prompt(url, model)},
        ],
        stream=True,
    )
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response


In [ ]:
url_input = gr.Textbox(
    label="Recipe or cookbook page URL including http:// or https://"
)
model_selector = gr.Dropdown(
    ["gpt-4.1-mini", "gpt-5.4-nano"], label="Select model", value="gpt-5.4-nano"
)
message_output = gr.Markdown(
    label="Response:"
)  # going to return the result in markdown format

view = gr.Interface(
    fn=stream_cookbook,
    title="Recipe Cookbook Generator for beginners",
    inputs=[url_input, model_selector],
    outputs=[message_output],
    examples=[
        ["https://www.budgetbytes.com/", "gpt-4.1-mini"],
        [
            "https://goodcheapeats.com/category/main-dishes/",
            "gpt-5.4-nano",
        ],
    ],
    flagging_mode="never",
)
view.launch()


* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://goodcheapeats.com/category/main-dishes/ by calling gpt-5.4-nano
Found 3 relevant links
Selecting relevant links for https://www.budgetbytes.com/ by calling gpt-4.1-mini
Found 3 relevant links
